# Cow Replacement RL Experiment Notebook

Bu notebook `Sürüden_keep_cull` projesinin RL tarafını tek yerden anlamak, parametrelemek, çalıştırmak ve sonuçlarını okumak için hazırlanmıştır.

Amaçlar:

- Hangi RL/MDP modelinin ne işe yaradığını açık görmek.
- Eğitim/çözüm parametrelerini notebook içinden yönetmek.
- Seçilen modeli veya model grubunu çalıştırmak.
- Eğitim/çözüm sonrası oluşan policy, metric, artifact ve raporları aynı notebook içinde incelemek.

Kapsam notu: Bu proje canlı sistem kararı vermez. Buradaki modeller araştırma/prototip amaçlıdır; reward ve transition kalitesi güçlendirilmeden production policy olarak kullanılmamalıdır.


## 0. Model Aileleri

| Aile                    | Dosya                                       | Ne üretir?                                                  | Pratik anlamı                                          |
| ----------------------- | ------------------------------------------- | ----------------------------------------------------------- | ------------------------------------------------------ |
| Model-based MDP         | `models/model_based/mdp_value_iteration.py` | reward table, value table, optimal policy, scenario analizi | Ana referans karar çözümü                              |
| Tabular RL              | `models/tabular/q_learning.py`              | tabular Q-learning policy                                   | Empirical simulator üzerinde basit RL benchmark        |
| Tabular policy gradient | `models/tabular/ppo_smoke.py`               | PPO-style tabular policy                                    | PPO fikri için hafif sanity check                      |
| Approximate Q           | `models/approximate/dqn_style_fqi.py`       | MLP tabanlı fitted Q policy                                 | Fonksiyon yaklaşımıyla Q öğrenme deneyi                |
| Deep RL                 | `models/stable_baselines/ppo.py`            | SB3 PPO `.zip` modeli + eval sonuçları                      | Gerçek neural DRL policy eğitimi                       |
| Deep RL                 | `models/stable_baselines/dqn.py`            | SB3 DQN `.zip` modeli + eval sonuçları                      | Gerçek neural DRL Q-network eğitimi                    |
| Combo runner            | `models/stable_baselines/ppo_dqn.py`        | PPO ve DQN'i sırayla çalıştırır                             | Ayrı model değil, kolaylık runner'ı                    |
| Offline RL placeholder  | `models/offline/placeholders.py`            | not-eligible kaydı                                          | Logged action olmadığı için CQL/IQL/BCQ şu an koşulmaz |


In [ ]:
from pathlib import Path
import json
import platform
import sys
import time
from datetime import datetime

import pandas as pd
import yaml
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

cwd = Path.cwd().resolve()
if (cwd / "config" / "rl_env_rules.yaml").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "config" / "rl_env_rules.yaml").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path("/mnt/c/Users/Artun/Desktop/Dosyalar/github_repos/LaktaSens/KDS/Sürüden_keep_cull").resolve()

CODE_DIR = PROJECT_ROOT / "Model" / "rl" / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from model_registry import available_models, model_table, run_model
from training_environment import RLTrainingEnvironment, load_yaml, save_yaml

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

env = RLTrainingEnvironment(project_root=PROJECT_ROOT)
CONFIG_PATH = env.config_path
PARAMETERS_PATH = env.parameters_path

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CODE_DIR:", CODE_DIR)
print("Python:", sys.executable)
print("Platform:", platform.platform())


## 1. Registry ve Mevcut Model Listesi

Bu hücre notebook'un görebildiği tüm model/algoritma modüllerini listeler. `trainable_now=False` olanlar veri koşulu hazır olmadığı için çalıştırılmaz veya sadece placeholder sonucu üretir.


In [ ]:
models_df = model_table()
display(models_df)

MODEL_EXPLANATIONS = pd.DataFrame([
    {"model_key": "mdp_value_iteration", "category": "model_based_mdp", "what_it_does": "Empirical transition + reward table ile value iteration çözer.", "main_output": "optimal_policy_table.csv, mdp_value_iteration_result.csv", "artifact_type": "policy/value table"},
    {"model_key": "tabular_q_learning", "category": "tabular_rl", "what_it_does": "Simulator rollout ile state-action Q değerlerini tabular öğrenir.", "main_output": "rl_algorithm_comparison_results.csv, rl_algorithm_policy_table.csv", "artifact_type": "policy table"},
    {"model_key": "tabular_ppo_smoke", "category": "tabular_policy_gradient", "what_it_does": "PPO-style hafif tabular policy güncellemesi yapar.", "main_output": "rl_algorithm_comparison_results.csv, rl_algorithm_policy_table.csv", "artifact_type": "policy probability table"},
    {"model_key": "dqn_style_fqi_mlp", "category": "approximate_q", "what_it_does": "MLPRegressor ile fitted Q iteration çalıştırır.", "main_output": "rl_algorithm_comparison_results.csv, rl_algorithm_policy_table.csv", "artifact_type": "policy table"},
    {"model_key": "sb3_ppo", "category": "deep_rl", "what_it_does": "Gymnasium env üzerinde Stable-Baselines3 PPO eğitir.", "main_output": "sb3_training_results.csv, sb3_policy_table.csv", "artifact_type": "Model/rl/sb3/ppo_cow_replacement.zip"},
    {"model_key": "sb3_dqn", "category": "deep_rl", "what_it_does": "Gymnasium env üzerinde Stable-Baselines3 DQN eğitir.", "main_output": "sb3_training_results.csv, sb3_policy_table.csv", "artifact_type": "Model/rl/sb3/dqn_cow_replacement.zip"},
    {"model_key": "sb3_ppo_dqn", "category": "deep_rl_combo", "what_it_does": "PPO ve DQN'i sırayla çalıştırır.", "main_output": "sb3_* çıktıları", "artifact_type": "combo runner"},
    {"model_key": "cql/iql/bcq", "category": "offline_rl", "what_it_does": "Logged action olmadığı için not-eligible sonucu üretir.", "main_output": "rl_algorithm_comparison_results.csv", "artifact_type": "placeholder status"},
])
display(MODEL_EXPLANATIONS)


## 2. Girdi/Çıktı Sağlık Kontrolü

Bu kontrol model çalıştırmadan önce gerekli CSV/config dosyalarının varlığını ve mevcut output durumunu gösterir.


In [ ]:
rules = env.load_config()
required_inputs = {
    "mdp_state_table": rules["output"]["mdp_state_table_csv"],
    "empirical_transition_model": rules["output"]["empirical_transition_model_csv"],
    "reward_baseline": rules["input"]["reward_baseline_csv"],
    "snapshot_input": rules["input"]["snapshot_csv"],
}
input_health = []
for name, rel_path in required_inputs.items():
    path = PROJECT_ROOT / rel_path
    input_health.append({"name": name, "path": rel_path, "exists": path.exists(), "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else None})
display(pd.DataFrame(input_health))

summary = env.summarize_outputs()
display(summary if not summary.empty else pd.DataFrame([{"info": "Henüz output tablosu bulunamadı."}]))


## 3. Eğitim ve Çözüm Parametreleri

Aşağıdaki hücredeki değerleri değiştirerek MDP, tabular RL, fitted Q ve SB3 PPO/DQN eğitimlerini yönetebilirsin.

Notlar:

- SB3 PPO/DQN gerçek DRL eğitimidir; uzun timestep değerleri daha uzun sürer.
- `RUN_MODELS` içine hangi modeli koyarsan sadece onlar çalışır.
- Offline RL (`cql`, `iql`, `bcq`) şu an gerçek eğitim yapmaz; uygun veri olmadığı için not-eligible sonucu yazar.


In [ ]:
params = env.load_parameters()
params.setdefault("run", {})
params.setdefault("python", {})
params.setdefault("overrides", {})
params.setdefault("profiles", {})

# Hangi modeller çalıştırılacak?
RUN_MODELS = [
    "mdp_value_iteration",
    "tabular_q_learning",
    "tabular_ppo_smoke",
    "dqn_style_fqi_mlp",
]

# DRL profili seçimi:
# - quick_smoke: 5k timestep; altyapı/sanity check.
# - long_drl_100k: ilk ciddi PPO/DQN koşusu; önerilen varsayılan.
# - long_drl_500k: 100k sonuçları anlamlıysa daha ağır validasyon.
TRAINING_PROFILE = "long_drl_100k"
params["profile"] = TRAINING_PROFILE

# Gerçek DRL eğitimi. long_drl_100k seçiliyken bu hücreden sonra çalıştırma hücresi
# PPO ve DQN için 100k timestep koşar; süre makineye göre uzayabilir.
RUN_DEEP_RL = True
DEEP_RL_MODELS = ["sb3_ppo", "sb3_dqn"]  # Alternatif: ["sb3_ppo_dqn"]

# Python ortamları
params["python"]["default"] = "python3"
params["python"]["sb3"] = "/tmp/laktasens-rl-venv/bin/python"

# MDP parametreleri
params["overrides"]["mdp.discount_factor"] = 0.97
params["overrides"]["mdp.convergence_tolerance"] = 0.0001
params["overrides"]["mdp.max_iterations"] = 1000

# Tabular RL parametreleri
params["overrides"]["rl_experiments.algorithms.tabular_q_learning.episodes"] = 4000
params["overrides"]["rl_experiments.algorithms.tabular_q_learning.max_steps"] = 60
params["overrides"]["rl_experiments.algorithms.tabular_q_learning.alpha"] = 0.18
params["overrides"]["rl_experiments.algorithms.tabular_q_learning.epsilon_start"] = 0.35
params["overrides"]["rl_experiments.algorithms.tabular_q_learning.epsilon_end"] = 0.03

# Tabular PPO-style smoke parametreleri
params["overrides"]["rl_experiments.algorithms.tabular_ppo_smoke.iterations"] = 140
params["overrides"]["rl_experiments.algorithms.tabular_ppo_smoke.episodes_per_iteration"] = 24
params["overrides"]["rl_experiments.algorithms.tabular_ppo_smoke.learning_rate"] = 0.015

# Fitted Q / MLP parametreleri
params["overrides"]["rl_experiments.algorithms.dqn_style_fqi_mlp.fitted_iterations"] = 10
params["overrides"]["rl_experiments.algorithms.dqn_style_fqi_mlp.hidden_layer_sizes"] = [32]
params["overrides"]["rl_experiments.algorithms.dqn_style_fqi_mlp.max_iter"] = 250

# Profil override'ları manuel değerlerin üstüne uygulanır.
profile_cfg = params.get("profiles", {}).get(TRAINING_PROFILE, {})
profile_overrides = dict(profile_cfg.get("overrides", {}) or {})
params["overrides"].update(profile_overrides)

save_yaml(PARAMETERS_PATH, params)
env.apply_overrides(params["overrides"])

print("Parametreler yazıldı:", PARAMETERS_PATH)
print("Aktif DRL profili:", TRAINING_PROFILE)
print("Profil açıklaması:", profile_cfg.get("description", ""))
display(pd.DataFrame([{"key": k, "value": v} for k, v in params["overrides"].items()]))


## 4. Seçili Modelleri Çalıştır

Bu hücre `RUN_MODELS` listesindeki modelleri sırayla çalıştırır. Deep RL için `RUN_DEEP_RL=True` yaparsan SB3 PPO/DQN de çalışır.


In [ ]:
all_requested_models = list(RUN_MODELS)
if RUN_DEEP_RL:
    all_requested_models.extend(DEEP_RL_MODELS)

run_rows = []
start_all = time.time()
for model_key in all_requested_models:
    print("\n" + "=" * 90)
    print("MODEL:", model_key)
    started = time.time()
    try:
        results = run_model(
            model_key,
            env=env,
            default_python=params["python"].get("default", sys.executable),
            sb3_python=params["python"].get("sb3"),
            check=True,
        )
        status = "completed"
        returncodes = [r.returncode for r in results]
    except Exception as exc:
        status = "failed"
        returncodes = []
        print("ERROR:", repr(exc))
    elapsed = time.time() - started
    run_rows.append({"model_key": model_key, "status": status, "returncodes": returncodes, "elapsed_sec": round(elapsed, 2)})

run_log = pd.DataFrame(run_rows)
print("\nTotal elapsed sec:", round(time.time() - start_all, 2))
display(run_log)


## 5. Output Tablolarını Yükle

Bu hücre son çalıştırmadan sonra oluşan ana CSV çıktıları notebook içine alır.


In [ ]:
tables = env.load_output_tables()
summary = env.summarize_outputs()
display(summary)

for name, df in tables.items():
    print(f"{name}: {df.shape}")


## 6. MDP Policy Sonuçları

Ana referans çözümün kaç state için `keep`, kaç state için `cull` dediğini ve en güçlü karar marjlarını gösterir.


In [ ]:
mdp_policy = tables.get("mdp_policy", pd.DataFrame())
if mdp_policy.empty:
    display(Markdown("MDP policy çıktısı bulunamadı."))
else:
    display(mdp_policy["optimal_action"].value_counts().rename_axis("action").reset_index(name="state_count"))
    margin_cols = ["state_bucket_exact", "optimal_action", "state_value", "q_keep", "q_cull", "action_margin_cull_minus_keep", "transition_fallback_used", "transition_support", "risk_band", "milk_level_band", "health_risk_band"]
    display(Markdown("### En güçlü cull state'leri"))
    display(mdp_policy.sort_values("action_margin_cull_minus_keep", ascending=False)[margin_cols].head(15))
    display(Markdown("### En güçlü keep state'leri"))
    display(mdp_policy.sort_values("action_margin_cull_minus_keep", ascending=True)[margin_cols].head(15))


## 7. RL Algoritma Karşılaştırması

Tabular Q-learning, PPO-style smoke, fitted Q ve placeholder offline RL sonuçlarını MDP referansı ile birlikte gösterir.


In [ ]:
comparison = tables.get("algorithm_comparison", pd.DataFrame())
policy_table = tables.get("rl_algorithm_policy", pd.DataFrame())
if comparison.empty:
    display(Markdown("RL comparison çıktısı bulunamadı."))
else:
    display(comparison.sort_values(["run_status", "mean_initial_state_value"], ascending=[True, False], na_position="last"))

if not policy_table.empty:
    action_counts = policy_table.groupby(["algorithm", "selected_action"]).size().unstack(fill_value=0).reset_index()
    display(Markdown("### Algoritma bazlı policy action sayıları"))
    display(action_counts)


## 8. Deep RL: SB3 PPO/DQN Sonuçları

Bu bölüm gerçek DRL eğitim çıktılarını gösterir. `RUN_DEEP_RL=False` kaldıysa burada son kayıtlı sonuçlar gösterilir.


In [ ]:
sb3_results = tables.get("sb3_results", pd.DataFrame())
sb3_policy = tables.get("sb3_policy", pd.DataFrame())
sb3_episodes = tables.get("sb3_eval_episodes", pd.DataFrame())

if sb3_results.empty:
    display(Markdown("SB3 PPO/DQN sonucu bulunamadı veya henüz çalıştırılmadı."))
else:
    display(sb3_results.sort_values("mean_total_reward", ascending=False))

if not sb3_policy.empty:
    display(Markdown("### SB3 policy action sayıları"))
    display(sb3_policy.groupby(["algorithm", "selected_action"]).size().unstack(fill_value=0).reset_index())

if not sb3_episodes.empty:
    display(Markdown("### Eval episode özeti"))
    display(sb3_episodes.groupby("algorithm").agg(
        episodes=("episode", "count"),
        mean_reward=("total_reward", "mean"),
        median_reward=("total_reward", "median"),
        mean_steps=("steps", "mean"),
        mean_cull_action_rate=("cull_action_rate", "mean"),
    ).reset_index())

artifact_rows = []
for rel in ["Model/rl/sb3/ppo_cow_replacement.zip", "Model/rl/sb3/dqn_cow_replacement.zip"]:
    path = PROJECT_ROOT / rel
    artifact_rows.append({"artifact": rel, "exists": path.exists(), "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else None})
display(pd.DataFrame(artifact_rows))


## 9. Scenario Sensitivity

Reward parametreleri değiştiğinde MDP policy nasıl değişiyor, burada incelenir.


In [ ]:
scenario_results = tables.get("scenario_results", pd.DataFrame())
if scenario_results.empty:
    display(Markdown("Scenario analysis sonucu bulunamadı."))
else:
    cols = ["scenario", "description", "discount_factor", "keep_states", "cull_states", "cull_state_rate", "changed_states_vs_baseline", "mean_state_value"]
    display(scenario_results[cols].sort_values("scenario"))


## 10. Eğitim ve Policy Grafikleri

Bu bölüm, eğitim/çözüm koşularından çıkan CSV dosyalarını grafikleştirir. Grafikler notebook içinde gösterilir ve aynı zamanda `reports/rl_notebook_figures/` altına PNG olarak kaydedilir. SB3 için mevcut dosyalarda loss eğrisi değil eval episode çıktıları bulunduğu için grafikler eğitim sonrası değerlendirme ve policy davranışı odaklıdır.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.style.use("default")

FIGURE_DIR = PROJECT_ROOT / "reports" / "rl_notebook_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Hücre tek başına çalıştırılırsa tabloları yeniden yükle.
if "tables" not in globals() or not isinstance(tables, dict):
    tables = env.load_output_tables()

comparison = tables.get("algorithm_comparison", pd.DataFrame())
policy_table = tables.get("rl_algorithm_policy", pd.DataFrame())
mdp_policy = tables.get("mdp_policy", pd.DataFrame())
scenario_results = tables.get("scenario_results", pd.DataFrame())
sb3_results = tables.get("sb3_results", pd.DataFrame())
sb3_episodes = tables.get("sb3_eval_episodes", pd.DataFrame())
sb3_policy = tables.get("sb3_policy", pd.DataFrame())
sb3_curve = tables.get("sb3_training_curve", pd.DataFrame())
def _load_csv_if_exists(rel_path: str) -> pd.DataFrame:
    path = PROJECT_ROOT / rel_path
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

if sb3_curve.empty:
    sb3_curve = _load_csv_if_exists("Data/processed/rl/sb3_training_curve.csv")
reward_table = tables.get("mdp_reward", tables.get("mdp_reward_table", pd.DataFrame()))
simulation_results = tables.get("policy_simulation_results", pd.DataFrame())
if simulation_results.empty:
    simulation_results = _load_csv_if_exists("Data/processed/rl/policy_simulation_results.csv")
simulation_trace = tables.get("policy_simulation_trace", pd.DataFrame())
if simulation_trace.empty:
    simulation_trace = _load_csv_if_exists("Data/processed/rl/policy_simulation_trace.csv")

saved_figures = []


def _save_show(fig, filename: str, title: str | None = None):
    if title:
        fig.suptitle(title, fontsize=12, y=1.02)
    fig.tight_layout()
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=170, bbox_inches="tight")
    saved_figures.append(path.relative_to(PROJECT_ROOT).as_posix())
    plt.show()


def _bar_label(ax, fmt="{:.2f}", fontsize=8):
    for container in ax.containers:
        try:
            ax.bar_label(container, fmt=fmt, fontsize=fontsize)
        except Exception:
            pass

# 1) Run süresi: notebook içinde model çalıştırıldıysa göster.
if "run_log" in globals() and isinstance(run_log, pd.DataFrame) and not run_log.empty and "elapsed_sec" in run_log.columns:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    df = run_log.sort_values("elapsed_sec", ascending=True)
    ax.barh(df["model_key"], df["elapsed_sec"], color="#4C78A8")
    ax.set_xlabel("Süre (saniye)")
    ax.set_ylabel("Model")
    ax.set_title("Notebook Koşu Süreleri")
    _bar_label(ax, fmt="%.1f")
    _save_show(fig, "01_run_elapsed_seconds.png")
else:
    display(Markdown("`run_log` bulunmadı; modeller bu notebook oturumunda çalıştırılınca koşu süresi grafiği üretilecek."))

# 2) RL algoritma karşılaştırması: değer ve cull oranı.
if not comparison.empty:
    comp = comparison.copy()
    comp = comp[comp.get("run_status", "completed").eq("completed")].copy() if "run_status" in comp.columns else comp
    if not comp.empty:
        comp = comp.sort_values("mean_initial_state_value", ascending=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.34 * len(comp))))
        axes[0].barh(comp["algorithm"], comp["mean_initial_state_value"], color="#2F6B3F")
        axes[0].set_xlabel("Ortalama başlangıç state değeri")
        axes[0].set_title("Policy Değeri Karşılaştırması")
        axes[1].barh(comp["algorithm"], comp["cull_state_rate"], color="#B65F2A")
        axes[1].set_xlabel("Cull state oranı")
        axes[1].set_title("Policy Agresifliği")
        _save_show(fig, "02_rl_algorithm_value_and_cull_rate.png")
else:
    display(Markdown("`algorithm_comparison` tablosu bulunamadı."))

# 3) MDP policy dağılımı ve cull-margin uçları.
if not mdp_policy.empty:
    counts = mdp_policy["optimal_action"].value_counts().reindex(["keep", "cull"]).dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].bar(counts.index.astype(str), counts.values, color=["#4C78A8", "#E45756"][:len(counts)])
    axes[0].set_ylabel("State sayısı")
    axes[0].set_title("MDP Optimal Aksiyon Dağılımı")
    _bar_label(axes[0], fmt="%.0f")

    margin_col = "action_margin_cull_minus_keep"
    if margin_col in mdp_policy.columns:
        margin = mdp_policy[["state_bucket_exact", margin_col, "optimal_action"]].dropna().copy()
        top_keep = margin.nsmallest(8, margin_col).assign(group="keep güçlü")
        top_cull = margin.nlargest(8, margin_col).assign(group="cull güçlü")
        extremes = pd.concat([top_keep, top_cull], ignore_index=True)
        colors = extremes["optimal_action"].map({"keep": "#4C78A8", "cull": "#E45756"}).fillna("#999999")
        axes[1].barh(range(len(extremes)), extremes[margin_col], color=colors)
        axes[1].set_yticks(range(len(extremes)))
        axes[1].set_yticklabels([s[:42] + ("..." if len(s) > 42 else "") for s in extremes["state_bucket_exact"].astype(str)], fontsize=7)
        axes[1].axvline(0, color="black", linewidth=0.8)
        axes[1].set_xlabel("Q(cull) - Q(keep)")
        axes[1].set_title("En Net Keep/Cull State'leri")
    _save_show(fig, "03_mdp_policy_distribution_and_margin.png")

# 4) Risk bandına göre cull policy yoğunluğu.
if not mdp_policy.empty and {"risk_band", "optimal_action"}.issubset(mdp_policy.columns):
    risk_policy = (
        mdp_policy.assign(is_cull=mdp_policy["optimal_action"].eq("cull").astype(int))
        .groupby("risk_band", dropna=False)
        .agg(states=("optimal_action", "size"), cull_rate=("is_cull", "mean"), mean_value=("state_value", "mean"))
        .reset_index()
    )
    risk_policy["risk_band"] = risk_policy["risk_band"].astype(str)
    fig, ax1 = plt.subplots(figsize=(9, 4.5))
    ax1.bar(risk_policy["risk_band"], risk_policy["cull_rate"], color="#E45756", alpha=0.78)
    ax1.set_ylabel("Cull oranı")
    ax1.set_xlabel("Risk bandı")
    ax1.set_title("Risk Bandına Göre MDP Cull Eğilimi")
    ax1.tick_params(axis="x", rotation=35)
    ax2 = ax1.twinx()
    ax2.plot(risk_policy["risk_band"], risk_policy["states"], marker="o", color="#4C78A8")
    ax2.set_ylabel("State sayısı")
    _save_show(fig, "04_mdp_cull_rate_by_risk_band.png")

# 5) Scenario sensitivity grafikleri.
if not scenario_results.empty:
    scen = scenario_results.sort_values("changed_states_vs_baseline", ascending=True).copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4.5, 0.36 * len(scen))))
    axes[0].barh(scen["scenario"], scen["cull_states"], color="#E45756")
    axes[0].set_title("Cull State Sayısı")
    axes[0].set_xlabel("Cull state")
    axes[1].barh(scen["scenario"], scen["changed_states_vs_baseline"], color="#F58518")
    axes[1].set_title("Baseline'a Göre Değişen State")
    axes[1].set_xlabel("State sayısı")
    axes[2].barh(scen["scenario"], scen["mean_state_value"], color="#54A24B")
    axes[2].set_title("Ortalama State Value")
    axes[2].set_xlabel("Value")
    _save_show(fig, "05_scenario_sensitivity.png")
else:
    display(Markdown("`scenario_results` tablosu bulunamadı."))

# 6) Stable-Baselines3 değerlendirme ve eğitim eğrisi grafikleri.
if not sb3_results.empty or not sb3_episodes.empty or not sb3_curve.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes = axes.ravel()
    if not sb3_results.empty:
        sb3 = sb3_results.sort_values("mean_total_reward", ascending=False)
        axes[0].bar(sb3["algorithm"], sb3["mean_total_reward"], color="#4C78A8")
        axes[0].set_title("Final Eval Ortalama Reward")
        axes[0].set_ylabel("Reward")
        _bar_label(axes[0], fmt="%.1f")
        if "policy_cull_state_rate" in sb3.columns:
            axes[1].bar(sb3["algorithm"], sb3["policy_cull_state_rate"], color="#E45756")
            axes[1].set_title("Final Policy Cull State Oranı")
            axes[1].set_ylabel("Cull oranı")
            _bar_label(axes[1], fmt="%.2f")
    if not sb3_curve.empty and {"algorithm", "timestep", "mean_total_reward"}.issubset(sb3_curve.columns):
        for alg, grp in sb3_curve.groupby("algorithm"):
            grp = grp.sort_values("timestep")
            axes[2].plot(grp["timestep"], grp["mean_total_reward"], marker="o", label=alg)
            if "mean_cull_action_rate" in grp.columns:
                axes[3].plot(grp["timestep"], grp["mean_cull_action_rate"], marker="o", label=alg)
        axes[2].set_title("Training Curve: Eval Reward")
        axes[2].set_xlabel("Timestep")
        axes[2].set_ylabel("Mean eval reward")
        axes[2].legend()
        axes[3].set_title("Training Curve: Eval Cull Action Rate")
        axes[3].set_xlabel("Timestep")
        axes[3].set_ylabel("Cull action rate")
        axes[3].legend()
    elif not sb3_episodes.empty:
        for alg, grp in sb3_episodes.groupby("algorithm"):
            grp = grp.sort_values("episode")
            rolling = grp["total_reward"].rolling(10, min_periods=1).mean()
            axes[2].plot(grp["episode"], rolling, label=alg)
        axes[2].set_title("Final Eval Reward - 10 Episode Rolling Mean")
        axes[2].set_xlabel("Episode")
        axes[2].set_ylabel("Reward")
        axes[2].legend()
        axes[3].axis("off")
    _save_show(fig, "06_sb3_eval_summary.png")

if not sb3_episodes.empty:
    algorithms = list(sb3_episodes["algorithm"].dropna().unique())
    data = [sb3_episodes.loc[sb3_episodes["algorithm"].eq(a), "total_reward"].dropna() for a in algorithms]
    if data:
        fig, ax = plt.subplots(figsize=(8, 4.8))
        ax.boxplot(data, labels=algorithms, showmeans=True)
        ax.set_title("SB3 Eval Episode Reward Dağılımı")
        ax.set_ylabel("Total reward")
        _save_show(fig, "07_sb3_eval_reward_distribution.png")

# 7) Reward table: keep/cull reward dağılımı ve reward bileşenleri.
if not reward_table.empty and {"action_name", "reward"}.issubset(reward_table.columns):
    actions = list(reward_table["action_name"].dropna().unique())
    data = [reward_table.loc[reward_table["action_name"].eq(a), "reward"].dropna() for a in actions]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    if data:
        axes[0].boxplot(data, labels=actions, showmeans=True)
        axes[0].set_title("Reward Dağılımı: Keep vs Cull")
        axes[0].set_ylabel("Reward")
    comp_cols = [
        "milk_revenue_month",
        "maintenance_cost_month",
        "health_risk_cost_month",
        "reproduction_risk_cost_month",
        "expected_culling_risk_cost_month",
    ]
    comp_cols = [c for c in comp_cols if c in reward_table.columns]
    keep_rewards = reward_table[reward_table["action_name"].eq("keep")]
    if comp_cols and not keep_rewards.empty:
        means = keep_rewards[comp_cols].mean().sort_values()
        axes[1].barh(means.index, means.values, color="#72B7B2")
        axes[1].set_title("Keep Reward Ortalama Bileşenleri")
        axes[1].set_xlabel("Aylık değer")
    _save_show(fig, "08_reward_distribution_and_components.png")

# 8) Policy simulation sonuçları.
if not simulation_results.empty:
    sim = simulation_results.sort_values("mean_total_reward", ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4.5, 0.35 * len(sim))))
    axes[0].barh(sim["policy"], sim["mean_total_reward"], color="#4C78A8")
    axes[0].set_title("Simülasyon Ortalama Total Reward")
    axes[0].set_xlabel("Reward")
    terminal_cols = [c for c in ["agent_cull_episodes", "observed_cull_episodes", "end_observation_episodes", "max_episode_episodes", "other_terminal_episodes"] if c in sim.columns]
    if terminal_cols:
        left = np.zeros(len(sim))
        colors = ["#E45756", "#F58518", "#54A24B", "#4C78A8", "#B279A2"]
        for idx, col in enumerate(terminal_cols):
            axes[1].barh(sim["policy"], sim[col], left=left, label=col.replace("_episodes", ""), color=colors[idx % len(colors)])
            left = left + sim[col].fillna(0).to_numpy()
        axes[1].set_title("Terminal Neden Dağılımı")
        axes[1].set_xlabel("Episode sayısı")
        axes[1].legend(fontsize=7)
    _save_show(fig, "09_policy_simulation_summary.png")

# 9) Policy trace: step bazlı reward trendi.
if not simulation_trace.empty and {"policy", "step", "reward"}.issubset(simulation_trace.columns):
    trend = simulation_trace.groupby(["policy", "step"], as_index=False).agg(mean_reward=("reward", "mean"))
    fig, ax = plt.subplots(figsize=(10, 5))
    for policy, grp in trend.groupby("policy"):
        grp = grp.sort_values("step")
        ax.plot(grp["step"], grp["mean_reward"], label=policy)
    ax.set_title("Simülasyon Step Bazlı Ortalama Reward")
    ax.set_xlabel("Step / ay")
    ax.set_ylabel("Ortalama reward")
    ax.legend(fontsize=8)
    _save_show(fig, "10_policy_trace_mean_reward_by_step.png")

if saved_figures:
    display(Markdown("### Kaydedilen grafikler\n" + "\n".join([f"- `{p}`" for p in saved_figures])))
else:
    display(Markdown("Grafik üretilecek uygun çıktı tablosu bulunamadı. Önce model koşularını çalıştırın veya output CSV'lerini üretin."))


### 10.1 Kaydedilmiş Grafik Galerisi

Aşağıdaki görseller `reports/rl_notebook_figures/` altında saklanır. Üstteki grafik hücresini tekrar çalıştırırsan bu dosyalar güncellenir.

![RL algoritma değer ve cull oranı](../reports/rl_notebook_figures/02_rl_algorithm_value_and_cull_rate.png)

![MDP policy dağılımı ve margin](../reports/rl_notebook_figures/03_mdp_policy_distribution_and_margin.png)

![Risk bandına göre MDP cull oranı](../reports/rl_notebook_figures/04_mdp_cull_rate_by_risk_band.png)

![Scenario sensitivity](../reports/rl_notebook_figures/05_scenario_sensitivity.png)

![SB3 eval summary](../reports/rl_notebook_figures/06_sb3_eval_summary.png)

![SB3 eval reward dağılımı](../reports/rl_notebook_figures/07_sb3_eval_reward_distribution.png)

![Reward dağılımı ve bileşenleri](../reports/rl_notebook_figures/08_reward_distribution_and_components.png)

![Policy simulation summary](../reports/rl_notebook_figures/09_policy_simulation_summary.png)

![Policy trace step reward](../reports/rl_notebook_figures/10_policy_trace_mean_reward_by_step.png)


## 11. Rapor Dosyalarını Önizle

Markdown raporlarının kısa önizlemesini gösterir. Detay için ilgili dosyayı açabilirsin.


In [ ]:
REPORTS = [
    "reports/mdp_policy_decision_report.md",
    "reports/mdp_scenario_analysis_report.md",
    "reports/rl_algorithm_comparison_report.md",
    "reports/sb3_ppo_dqn_training_report.md",
]

for rel in REPORTS:
    path = PROJECT_ROOT / rel
    display(Markdown(f"### `{rel}`"))
    if not path.exists():
        display(Markdown("Bulunamadı."))
        continue
    text = path.read_text(encoding="utf-8", errors="replace")
    preview = "\n".join(text.splitlines()[:45])
    display(Markdown(preview))


## 12. Run Manifest Kaydet

Bu hücre, kullanılan parametreleri ve son çıktı özetini timestamp'li bir JSON manifest olarak kaydeder. Deney tekrar üretilebilirliği için kullanılır.


In [ ]:
manifest = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "python": sys.executable,
    "requested_models": all_requested_models if "all_requested_models" in globals() else [],
    "run_log": run_log.to_dict("records") if "run_log" in globals() else [],
    "parameters_path": str(PARAMETERS_PATH.relative_to(PROJECT_ROOT)),
    "config_path": str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
    "overrides": params.get("overrides", {}) if "params" in globals() else {},
    "output_summary": summary.to_dict("records") if "summary" in globals() and not summary.empty else [],
}
manifest_dir = PROJECT_ROOT / "reports" / "rl_notebook_manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_dir / f"rl_notebook_manifest_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifest written:", manifest_path.relative_to(PROJECT_ROOT))


## 13. Okuma Rehberi

Sonuçları yorumlarken şu sırayı izle:

1. `mdp_policy`: Ana referans politika. `keep/cull` dağılımı ve Q marjları mantıklı mı?
2. `algorithm_comparison`: Basit RL benchmark'ları MDP referansına yakın mı, tamamen farklı mı?
3. `sb3_results`: PPO/DQN gerçek eğitimde reward ve action dağılımı açısından makul mü?
4. `scenario_results`: Milk price, replacement cost gibi reward varsayımları kararı değiştiriyor mu?
5. Raporlar: En güçlü cull/keep state'leri ve sensitivity notlarını oku.

Akademik not: PPO/DQN burada gerçek DRL algoritmalarıdır; fakat simülatör empirical transition + prototip reward üstünde kurulduğu için sonuçlar production policy değil, araştırma çıktısıdır.


## 14. Teslim Paketi — Çok-Seed Eğitim, Standart Grafikler ve Sonuçlar

Bu bölüm teslim isterlerine uygun pipeline'ı **notebook'tan tetikler ve sonuçları burada gösterir**:
- `>=5 seed` (42, 123, 2024, 7, 99), her grafik ortalama ± std bandı
- ham log (`teslim/sonuclar/loglar/`), `sonuclar.csv` (seed × metrik)
- 5 zorunlu standart grafik (öğrenme / eval / loss+heatmap / hiperparametre / baseline)

Ana ajanlar: **Tabular Q-Learning** + **SB3 DQN**. Referans: MDP value iteration. Baseline: random / keep_all / cull_all / risk_threshold.

Profil seçimi: `smoke` (dakikalar, sanity) — `full` (asıl teslim; DQN 100k timestep × 5 seed, ~25-35 dk).

In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display
import pandas as pd

# PROJECT_ROOT yukarıdaki hücrelerde tanımlı; tek başına çalıştırılırsa yeniden bul.
if "PROJECT_ROOT" not in globals():
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / "config" / "rl_env_rules.yaml").exists() else cwd.parent

TESLIM = PROJECT_ROOT / "teslim"
TESLIM_KOD = TESLIM / "kod"

# === Profil seçimi ===
TESLIM_PROFILE = "smoke"          # "smoke" | "full"
EVAL_EPISODES = 200 if TESLIM_PROFILE == "smoke" else 500


def run_teslim(script, *args):
    """teslim/kod altındaki bir scripti çalıştırır; çıktıyı buraya basar."""
    cmd = [sys.executable, str(TESLIM_KOD / script), *map(str, args)]
    print("$", " ".join(str(c) for c in cmd))
    res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, capture_output=True,
                         encoding="utf-8", errors="replace")
    if res.stdout:
        print(res.stdout[-4000:])
    if res.returncode != 0:
        print("STDERR:", (res.stderr or "")[-2500:])
    print(f"[{script} exit={res.returncode}]")
    return res.returncode


print("Profil:", TESLIM_PROFILE, "| eval_episodes:", EVAL_EPISODES)
print("Seedler:", (TESLIM_KOD / "seeds.txt").read_text().split())


### 14.1 Çok-seed eğitim (Q-learning + DQN) — ham log üretir

In [ ]:
run_teslim("train.py", "--algos", "qlearning,dqn", "--profile", TESLIM_PROFILE)

### 14.2 Hiperparametre taraması (epsilon_start × alpha, Grafik 4 verisi)

In [ ]:
run_teslim("sweep.py", "--profile", TESLIM_PROFILE)

### 14.3 Değerlendirme + `sonuclar.csv` (seed × metrik özeti)

In [ ]:
run_teslim("evaluate.py", "--eval-episodes", EVAL_EPISODES)

agg_path = TESLIM / "sonuclar" / "sonuclar_agg.csv"
if agg_path.exists():
    agg = pd.read_csv(agg_path).sort_values("eval_return_mean_mean", ascending=False)
    display(Markdown("**Seedler arası ortalama getiri (yüksek = iyi):**"))
    display(agg[["algorithm", "kind", "eval_return_mean_mean", "eval_return_mean_std",
                 "cull_episode_rate_mean", "n_seeds"]])


### 14.4 Beş zorunlu standart grafik (inline) + yorumlar

In [ ]:
run_teslim("plotting.py")

GRAF = TESLIM / "sunum" / "grafikler"
PNGS = [
    "01_ogrenme_egrisi.png",
    "02_eval_egrisi.png",
    "03_loss_ve_policy_heatmap.png",
    "04_hiperparametre_duyarlilik.png",
    "05_baseline_karsilastirma.png",
]
for png in PNGS:
    path = GRAF / png
    if path.exists():
        display(Markdown(f"#### {png}"))
        display(Image(filename=str(path)))

yorum_path = GRAF / "yorumlar.md"
if yorum_path.exists():
    display(Markdown("---\n### Grafik Yorumları (4 cümlelik model — taslak)\n"))
    display(Markdown(yorum_path.read_text(encoding="utf-8")))


### 14.5 Teslim Kabul-Şartı Checklist

| Kabul şartı | Durum |
|---|---|
| `sonuclar/loglar/` altında ham CSV log | ✅ `teslim/sonuclar/loglar/*_seed*.csv` |
| 5 grafik PNG, y-ekseni **episode getirisi** | ✅ `teslim/sunum/grafikler/01..05_*.png` |
| Her grafik 4 cümlelik yorum | ✅ `grafikler/yorumlar.md` (raporda cilalanacak) |
| `requirements.txt` (==), `seeds.txt` (≥5), `config.yaml`, `run_all.sh` | ✅ `teslim/kod/` |
| Baseline (random/heuristik) karşılaştırma | ✅ Grafik 5 |
| Yasaklar: `import gym`, `np.random.seed/choice` | ✅ yok (gymnasium + default_rng) |

> **Not:** Yukarıdaki sonuçlar `smoke` profilinden ise sanity amaçlıdır. Asıl teslim için
> `TESLIM_PROFILE = "full"` yapıp 14.1–14.4'ü yeniden çalıştırın (DQN 100k × 5 seed).
> Final rapor yorumları **`full` koşusunun** sayılarından yazılmalıdır.

### 14.6 Teslim Arşivini Kur + Kabul-Şartlarını Doğrula

`255172008_ArtunKARA.zip` arşivini kurar (rapor/sunum/kod/sonuclar) ve kabul-şartı checklist'ini çalıştırır. Ara ağırlık/`__pycache__`/SB3 ham dizinler hariç tutulur.

In [ ]:
res = subprocess.run([sys.executable, str(TESLIM / "build_package.py")],
                     cwd=str(PROJECT_ROOT), text=True, capture_output=True,
                     encoding="utf-8", errors="replace")
print(res.stdout[-5000:])
if res.returncode != 0:
    print("STDERR:", (res.stderr or "")[-2000:])
zip_path = TESLIM / "dist" / "255172008_ArtunKARA.zip"
if zip_path.exists():
    mb = zip_path.stat().st_size / 1024 / 1024
    display(Markdown(f"**Arşiv hazır:** `{zip_path.relative_to(PROJECT_ROOT)}` ({mb:.2f} MB) "
                     f"+ ayrı `255172008_video.mp4` (bkz. `teslim/VIDEO_CEKIM_REHBERI.md`)."))
